# Задача 2. NER брендов в запросе

Найдите упоминания брендов в сообщении и сопоставьте их со справочником.

Для `хочу футболку моспошиф и кеды нью баланс` ожидаем

```python
[
    {"text": "моспошиф", "start": 14, "end": 22, "brand_id": "b_mos"},
    {"text": "нью баланс", "start": 30, "end": 40, "brand_id": "b_nb"},
]
```

Нужно учесть опечатки, словоформы, транслитерацию и многословные названия. При одинаковом сходстве выбираем более популярный бренд. Результат содержит непересекающиеся фрагменты и по одному ID для каждого.

ID и частоты заданы для упражнения. Вариант `new` есть у двух брендов.

## Установка зависимостей

In [ ]:
!pip install rapidfuzz nltk Unidecode

In [ ]:
import re
import unicodedata
import math
from functools import lru_cache
from rapidfuzz.distance import Levenshtein
from nltk.stem.snowball import SnowballStemmer
from unidecode import unidecode

# ID, название, варианты написания, условная популярность.
brands = [
    ("b_mos", "Моспошив", ["моспошив", "mosposhiv"], 70),
    ("b_nb", "New Balance", ["new balance", "нью баланс", "new"], 650),
    ("b_ne", "New Era", ["new era", "нью эра", "new"], 250),
    ("b_nike", "Nike", ["nike", "найк"], 1100),
    ("b_tnf", "The North Face", ["the north face", "норт фейс"], 600),
    ("b_gold", "Золотое Яблоко", ["золотое яблоко"], 500),
    ("b_samsung", "Samsung", ["samsung", "самсунг"], 1200),
    ("b_samson", "Samson", ["samson", "самсон"], 40),
]

## 1. Подготовка текста

После нормализации применяем русский Snowball-стеммер и транслитерацию в латиницу. Так же обрабатываем варианты названий в справочнике.

Посмотрите на `Золотое Яблоко` и `Золотого Яблока`. Для `New Balance` отдельно записан вариант `нью баланс` — одной транслитерации для такого совпадения недостаточно.

In [ ]:
stemmer = SnowballStemmer("russian")

def normalize(text):
    text = unicodedata.normalize("NFKC", text).casefold().replace("ё", "е")
    return re.sub(r"[\W_]+", " ", text).strip()

@lru_cache(maxsize=1024)
def search_form(text):
    stems = [stemmer.stem(word) for word in normalize(text).split()]
    return unidecode(" ".join(stems))

for text in ["МОСПОШИВ!", "mosposhiv", "Золотое Яблоко", "Золотого Яблока", "нью баланс"]:
    print(f"{text:20} → {search_form(text)}")
assert search_form("Золотого Яблока") == search_form("Золотое Яблоко")

aliases = [(brand_id, name, search_form(alias), popularity)
           for brand_id, name, variants, popularity in brands for alias in variants]
max_log_popularity = max(math.log1p(popularity) for _, _, _, popularity in brands)

## 2. Словные n-граммы · TODO 1

Перебираем окна из 1–3 слов. Три слова — длина самого длинного названия в справочнике.

Заполните тело цикла. Получите начало первого токена, конец последнего и добавьте `(start, end, message[start:end])` в `spans`. Границы относятся к исходному сообщению, правая граница исключительная.

In [ ]:
def word_spans(message, max_words=3):
    tokens = list(re.finditer(r"[^\W_]+", message))
    spans = []
    for first in range(len(tokens)):
        for last in range(first, min(first + max_words, len(tokens))):
            # TODO 1 — границы и тройка с исходным фрагментом.
            raise NotImplementedError("TODO 1")
    return spans

assert (5, 16, "New Balance") in word_spans("хочу New Balance!")
assert word_spans("...") == []
print(word_spans("кеды нью баланс"))

## 3. Кандидаты для каждого фрагмента

Редакционное сходство равно `1 − distance / max_length`. Порог — 0.8. Для строк длиной до трёх символов допускаем только точное совпадение.

Сравниваем фрагмент с вариантами названий той же длины в словах. Например, `new balance и` содержит три слова и пропускается при сравнении с `new balance`.

In [ ]:
def span_candidates(span, min_similarity=0.8):
    start, end, text = span
    query = search_form(text)
    candidates = []
    for brand_id, name, alias, popularity in aliases:
        if len(query.split()) != len(alias.split()):
            continue
        similarity = Levenshtein.normalized_similarity(query, alias)
        threshold = 1.0 if min(len(query), len(alias)) <= 3 else min_similarity
        if similarity >= threshold:
            candidates.append(dict(start=start, end=end, text=text, brand_id=brand_id,
                                   name=name, similarity=similarity, popularity=popularity))
    return candidates

print(span_candidates((0, 8, "моспошиф")))

## 4. Популярность бренда

Вычислите `similarity + weight × log(1 + popularity) / max_log_popularity`. Для логарифма используйте `math.log1p`.

При `weight=0.03` популярность добавляет к оценке до 0.03. Сравните кандидатов для `new`. Затем увеличьте вес и проверьте, какой бренд побеждает для `Samson`.

In [ ]:
def ranking_score(candidate, weight=0.03):
    # TODO 2 — нормированный логарифм популярности и итоговый score.
    raise NotImplementedError("TODO 2")

assert ranking_score({"similarity": 0.9, "popularity": 0}) == 0.9
assert ranking_score({"similarity": 0.9, "popularity": 1200}, weight=0) == 0.9
new_candidates = sorted(span_candidates((0, 3, "new")), key=ranking_score, reverse=True)
assert new_candidates[0]["brand_id"] == "b_nb"
print([(c["name"], round(ranking_score(c), 3)) for c in new_candidates])

## 5. Пересекающиеся упоминания

В `new balance` найдутся два фрагмента — `new` и `new balance`. Рассматриваем сначала длинные фрагменты, затем кандидатов с большей оценкой. Уже занятые участки пропускаем.

Реализуйте `overlaps`. Два интервала пересекаются, если начало каждого меньше конца другого. Интервалы `[0, 3)` и `[3, 7)` не пересекаются.

In [ ]:
def overlaps(left, right):
    # TODO 3 — два условия пересечения интервалов.
    raise NotImplementedError("TODO 3")

assert overlaps({"start": 0, "end": 11}, {"start": 0, "end": 3})
assert not overlaps({"start": 0, "end": 3}, {"start": 3, "end": 7})
assert not overlaps({"start": 9, "end": 12}, {"start": 0, "end": 3})

In [ ]:
def recognize_brands(message, weight=0.03):
    candidates = [candidate for span in word_spans(message) for candidate in span_candidates(span)]
    ranked = sorted(candidates, key=lambda c: (
        -(c["end"] - c["start"]), -ranking_score(c, weight), c["start"], c["brand_id"]))
    selected = []
    for candidate in ranked:
        if not any(overlaps(candidate, previous) for previous in selected):
            selected.append(candidate)
    return sorted(selected, key=lambda c: c["start"])

message = "хочу футболку моспошиф и кеды нью баланс"
entities = recognize_brands(message)
print([{key: e[key] for key in ["text", "start", "end", "brand_id"]} for e in entities])
assert [e["brand_id"] for e in entities] == ["b_mos", "b_nb"]
assert all(message[e["start"]:e["end"]] == e["text"] for e in entities)
assert recognize_brands("хочу обычную футболку") == []
assert recognize_brands("") == []

## 6. Оценка NER

Считаем Precision и Recall по множествам `(start, end, brand_id)`. Для совпадения нужны правильные границы и ID.

Разметка ниже содержит текст упоминаний, по которому получаем эталонные границы. В `recognize_brands` передаём сообщение целиком.

In [ ]:
cases = [
    ("хочу футболку моспошиф", [("моспошиф", "b_mos")]),
    ("кеды нью баланс", [("нью баланс", "b_nb")]),
    ("футболка mosposhiv", [("mosposhiv", "b_mos")]),
    ("из Золотого Яблока", [("Золотого Яблока", "b_gold")]),
    ("куртка the north face", [("the north face", "b_tnf")]),
    ("кеды new", [("new", "b_nb")]),
    ("колонка Samson", [("Samson", "b_samson")]),
    ("nike и моспошив", [("nike", "b_nike"), ("моспошив", "b_mos")]),
    ("нужна обычная футболка", []),
]
true_positive = predicted_count = gold_count = 0
for text, mentions in cases:
    gold = {(text.index(mention), text.index(mention) + len(mention), bid) for mention, bid in mentions}
    predicted = {(e["start"], e["end"], e["brand_id"]) for e in recognize_brands(text)}
    true_positive += len(gold & predicted)
    predicted_count += len(predicted)
    gold_count += len(gold)
    print("OK" if gold == predicted else "РАЗОБРАТЬ", text)
print({"Precision": true_positive / predicted_count if predicted_count else 0,
       "Recall": true_positive / gold_count if gold_count else 0})